In [8]:
"""
Script para gerar predições no conjunto de teste.
"""
import pandas as pd
import numpy as np
import joblib
import ast
from collections import defaultdict
import logging
import os

# Configurar logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def parse_inputs(inputs_series):
    """Converte strings de inputs para dicionários."""
    return inputs_series.apply(ast.literal_eval)

def extract_features_from_test(df):
    """
    Extrai features do test.csv (mesmas features usadas no treino).
    Tratar dados incompletos (keyup ausentes na base test.csv).
    """
    features = []
    linhas_sem_keyup = 0
    linhas_sem_keydown = 0
    
    for idx, row in df.iterrows():
        try:
            data = row['inputs_parsed']
            keyboard_data = data.get('keyboard', {})
            
            # TRATAMENTO: verificar se keydown/keyup existem
            keydown = keyboard_data.get('keydown', [])
            keyup = keyboard_data.get('keyup', [])
            
            # Log para debug (opcional - remover em produção)
            if len(keyup) == 0 and len(keydown) > 0:
                linhas_sem_keyup += 1
            if len(keydown) == 0:
                linhas_sem_keydown += 1
            
            # Dicionário para armazenar features
            feat = {}
            
            # Features básicas
            feat['length'] = data.get('length', len(keydown))
            feat['n_keydown'] = len(keydown)
            feat['n_keyup'] = len(keyup)
            feat['n_unique_codes'] = len(set([e['code'] for e in keydown])) if keydown else 0
            
            # Tempos totais
            if keydown:
                max_keydown = max(e['tick'] for e in keydown)
                min_keydown = min(e['tick'] for e in keydown)
            else:
                max_keydown = 0
                min_keydown = 0
                
            if keyup:
                max_keyup = max(e['tick'] for e in keyup)
            else:
                max_keyup = 0
            
            feat['total_time'] = max(max_keyup, max_keydown)
            feat['first_tick'] = min_keydown if keydown else 0
            
            # Calcular hold times (APENAS se houver keydown E keyup)
            hold_times = []
            if keydown and keyup:
                keydown_queue = defaultdict(list)
                
                for event in sorted(keydown, key=lambda x: x['tick']):
                    keydown_queue[event['code']].append(event['tick'])
                
                for event in sorted(keyup, key=lambda x: x['tick']):
                    code = event['code']
                    if keydown_queue[code]:
                        press_time = keydown_queue[code].pop(0)
                        hold_time = event['tick'] - press_time
                        hold_times.append(hold_time)
            
            # Estatísticas de hold times
            if hold_times:
                feat['hold_mean'] = np.mean(hold_times)
                feat['hold_std'] = np.std(hold_times)
                feat['hold_min'] = np.min(hold_times)
                feat['hold_max'] = np.max(hold_times)
                feat['hold_median'] = np.median(hold_times)
                feat['hold_q25'] = np.percentile(hold_times, 25)
                feat['hold_q75'] = np.percentile(hold_times, 75)
                feat['hold_cv'] = feat['hold_std'] / feat['hold_mean'] if feat['hold_mean'] > 0 else 0
            else:
                # Valores default para quando não há hold times
                feat['hold_mean'] = 0
                feat['hold_std'] = 0
                feat['hold_min'] = 0
                feat['hold_max'] = 0
                feat['hold_median'] = 0
                feat['hold_q25'] = 0
                feat['hold_q75'] = 0
                feat['hold_cv'] = 0
            
            # Calcular flight times (APENAS se houver eventos)
            flight_times = []
            all_events = []
            
            for e in keydown:
                all_events.append(('down', e['code'], e['tick']))
            for e in keyup:
                all_events.append(('up', e['code'], e['tick']))
            
            if len(all_events) > 1:
                all_events.sort(key=lambda x: x[2])
                for i in range(1, len(all_events)):
                    flight_times.append(all_events[i][2] - all_events[i-1][2])
            
            if flight_times:
                feat['flight_mean'] = np.mean(flight_times)
                feat['flight_std'] = np.std(flight_times)
                feat['flight_min'] = np.min(flight_times)
                feat['flight_max'] = np.max(flight_times)
                feat['flight_median'] = np.median(flight_times)
                feat['flight_cv'] = feat['flight_std'] / feat['flight_mean'] if feat['flight_mean'] > 0 else 0
            else:
                feat['flight_mean'] = 0
                feat['flight_std'] = 0
                feat['flight_min'] = 0
                feat['flight_max'] = 0
                feat['flight_median'] = 0
                feat['flight_cv'] = 0
            
            # Press-press intervals (APENAS se houver keydown)
            press_press = []
            if len(keydown) > 1:
                keydown_times = [e['tick'] for e in sorted(keydown, key=lambda x: x['tick'])]
                for i in range(1, len(keydown_times)):
                    press_press.append(keydown_times[i] - keydown_times[i-1])
            
            if press_press:
                feat['press_press_mean'] = np.mean(press_press)
                feat['press_press_std'] = np.std(press_press)
                feat['press_press_cv'] = feat['press_press_std'] / feat['press_press_mean'] if feat['press_press_mean'] > 0 else 0
            else:
                feat['press_press_mean'] = 0
                feat['press_press_std'] = 0
                feat['press_press_cv'] = 0
            
            # Velocidade
            if feat['total_time'] > 0:
                feat['keys_per_second'] = feat['n_keydown'] / (feat['total_time'] / 1000)
            else:
                feat['keys_per_second'] = 0
            
            # Ratio
            if feat['flight_mean'] > 0:
                feat['hold_flight_ratio'] = feat['hold_mean'] / feat['flight_mean']
            else:
                feat['hold_flight_ratio'] = 0
            
            features.append(feat)
            
        except Exception as e:
            logging.error(f"Erro na linha {idx}: {e}")
            logging.error(f"Dados: {row.get('inputs', 'N/A')[:200]}...")  # Log para debug
            # Em caso de erro, adicionar features com zeros
            feat = {k: 0 for k in ['length', 'n_keydown', 'n_keyup', 'n_unique_codes', 
                                   'total_time', 'first_tick', 'hold_mean', 'hold_std',
                                   'hold_min', 'hold_max', 'hold_median', 'hold_q25',
                                   'hold_q75', 'hold_cv', 'flight_mean', 'flight_std',
                                   'flight_min', 'flight_max', 'flight_median', 'flight_cv',
                                   'press_press_mean', 'press_press_std', 'press_press_cv',
                                   'keys_per_second', 'hold_flight_ratio']}
            features.append(feat)
    
    # Log de estatísticas (opcional)
    if linhas_sem_keyup > 0:
        logging.info(f"⚠️ Linhas com keydown mas sem keyup: {linhas_sem_keyup}")
    if linhas_sem_keydown > 0:
        logging.info(f"⚠️ Linhas sem keydown: {linhas_sem_keydown}")
    
    return pd.DataFrame(features)

def generate_predictions():
    """
    Gera predições para o test.csv e salva result.csv
    """
    
    # 1. Carregar modelo e feature names
    logging.info("Carregando modelo...")
    model = joblib.load('random_forest_final.pkl') # ou './random_forest_final.pkl'
    feature_names = joblib.load('feature_names.pkl')
    
    logging.info(f"Modelo carregado. Features esperadas: {len(feature_names)}")
    
    # 2. Carregar test.csv
    logging.info("Carregando test.csv...")
    df_test = pd.read_csv('../../data/test.csv')
    logging.info(f"Test set shape: {df_test.shape}")
    
    # 3. Processar inputs
    logging.info("Processando inputs do teste...")
    df_test['inputs_parsed'] = parse_inputs(df_test['inputs'])
    
    # 4. Extrair features
    logging.info("Extraindo features...")
    df_test_features = extract_features_from_test(df_test)
    logging.info(f"Features extraídas: {df_test_features.shape}")
    
    # 5. Garantir mesma ordem das features do treino
    logging.info("Alinhando features com o treino...")
    
    # Verificar se todas as features necessárias existem
    missing_features = set(feature_names) - set(df_test_features.columns)
    if missing_features:
        logging.warning(f"Features faltantes no teste: {missing_features}")
        # Adicionar com zeros
        for feat in missing_features:
            df_test_features[feat] = 0
    
    # Selecionar apenas as features usadas no treino e na mesma ordem
    X_test = df_test_features[feature_names].copy()
    
    # Verificar valores nulos
    if X_test.isnull().any().any():
        logging.warning("Valores nulos encontrados. Preenchendo com 0...") #Não remover linhas do teste. O arquivo result.csv deve ter o MESMO número de linhas do test.csv
        X_test = X_test.fillna(0)
    
    logging.info(f"X_test shape final: {X_test.shape}")
    
    # 6. Gerar predições
    logging.info("Gerando predições...")
    y_pred = model.predict_proba(X_test)[:, 1]
    
    # 7. Criar result.csv
    logging.info("Salvando result.csv...")
    result = pd.DataFrame({'target': y_pred})
    result.to_csv('../../data/result.csv', index=False)
    
    # 8. Verificar ordem (salvar um log com primeiras linhas)
    logging.info("\nPrimeiras 10 linhas do result.csv:")
    logging.info("\n" + str(result.head(10)))
    
    # 9. Estatísticas das predições
    logging.info("\nEstatísticas das predições:")
    logging.info(f"Média: {y_pred.mean():.4f}")
    logging.info(f"Std: {y_pred.std():.4f}")
    logging.info(f"Min: {y_pred.min():.4f}")
    logging.info(f"Max: {y_pred.max():.4f}")
    logging.info(f"Mediana: {np.median(y_pred):.4f}")
    
    # 10. Comparar com sample_result.csv (se existir)
    sample_path = '../../data/sample_result.csv'
    if os.path.exists(sample_path):
        sample = pd.read_csv(sample_path)
        logging.info(f"\nSample result shape: {sample.shape}")
        logging.info(f"Sample result primeiras linhas:\n{sample.head()}")
    
    logging.info("\n✅ Arquivo result.csv gerado com sucesso!")
    
    return result

if __name__ == "__main__":
    generate_predictions()

2026-02-26 18:44:21,391 - INFO - Carregando modelo...
2026-02-26 18:44:21,405 - INFO - Modelo carregado. Features esperadas: 25
2026-02-26 18:44:21,407 - INFO - Carregando test.csv...
2026-02-26 18:44:21,453 - INFO - Test set shape: (17907, 1)
2026-02-26 18:44:21,453 - INFO - Processando inputs do teste...
2026-02-26 18:44:23,467 - INFO - Extraindo features...
2026-02-26 18:44:26,878 - INFO - ⚠️ Linhas com keydown mas sem keyup: 638
2026-02-26 18:44:27,131 - INFO - Features extraídas: (17907, 25)
2026-02-26 18:44:27,131 - INFO - Alinhando features com o treino...
2026-02-26 18:44:27,135 - INFO - X_test shape final: (17907, 25)
2026-02-26 18:44:27,135 - INFO - Gerando predições...
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    0.0s finished
2026-02-26 18:44:27,164 - INFO - Salvando result.csv...
2026-02-26 18:44:27,176 - INFO - 
Primei